In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

ValueError: Mountpoint must not already contain files

In [ ]:
dataset_path = "/content/drive/MyDrive/SkincareDataset/skin_type_classification_dataset"

In [ ]:
!pip install opencv-python pillow tqdm

In [ ]:
!rm -rf /content/drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
import os

print(os.listdir(dataset_path))

['skinalaysis_labeling_train1.xlsx', 'skinanalysis_valid1.xlsx', 'valid', 'test', 'train']


In [ ]:
train_path = os.path.join(dataset_path, "train")

print(os.listdir(train_path))

['normal', 'oily', 'combination', 'dry']


In [ ]:
from PIL import Image
import os

bad_files = []

for root, dirs, files in os.walk(dataset_path):

    for file in files:

        if file.lower().endswith((".jpg", ".jpeg", ".png")):

            path = os.path.join(root, file)

            try:

                img = Image.open(path)

                img.verify()

            except:

                bad_files.append(path)

print("Corrupted Images Found:", len(bad_files))

Corrupted Images Found: 0


In [ ]:
!pip install imagehash
from PIL import Image
import imagehash
import os
import shutil
from tqdm import tqdm

dataset_path = "/content/drive/MyDrive/SkincareDataset/skin_type_classification_dataset"

duplicate_folder = "/content/drive/MyDrive/SkincareDataset/Duplicates"

os.makedirs(duplicate_folder, exist_ok=True)

hashes = {}

duplicate_count = 0

for root, dirs, files in os.walk(dataset_path):

    for file in tqdm(files):

        if file.lower().endswith((".jpg",".jpeg",".png")):

            path = os.path.join(root,file)

            try:

                img = Image.open(path)

                img_hash = imagehash.average_hash(img)

                if img_hash in hashes:

                    duplicate_count += 1

                    shutil.move(
                        path,
                        os.path.join(
                            duplicate_folder,
                            f"dup_{duplicate_count}_{file}"
                        )
                    )

                else:

                    hashes[img_hash] = path

            except:

                pass

print("Duplicates Found:", duplicate_count)

In [ ]:
import cv2
import os
import shutil
from tqdm import tqdm

dataset_path = "/content/drive/MyDrive/SkincareDataset/skin_type_classification_dataset"

blurry_folder = "/content/drive/MyDrive/SkincareDataset/BlurryImages"

os.makedirs(blurry_folder, exist_ok=True)

threshold = 50

blurry_count = 0

for root, dirs, files in os.walk(dataset_path):

    for file in tqdm(files):

        if file.lower().endswith((".jpg",".jpeg",".png")):

            path = os.path.join(root,file)

            img = cv2.imread(path)

            if img is None:
                continue

            gray = cv2.cvtColor(
                img,
                cv2.COLOR_BGR2GRAY
            )

            score = cv2.Laplacian(
                gray,
                cv2.CV_64F
            ).var()

            if score < threshold:

                blurry_count += 1

                shutil.move(
                    path,
                    os.path.join(
                        blurry_folder,
                        f"blur_{blurry_count}_{file}"
                    )
                )

print("Blurry Images Found:", blurry_count)

In [ ]:
import os

print("Duplicates:", len(os.listdir("/content/drive/MyDrive/SkincareDataset/Duplicates")))
print("Blurry:", len(os.listdir("/content/drive/MyDrive/SkincareDataset/BlurryImages")))

In [ ]:
import os

train_path = os.path.join(dataset_path, "train")

for cls in os.listdir(train_path):

    class_path = os.path.join(train_path, cls)

    if os.path.isdir(class_path):

        print(
            cls,
            len(os.listdir(class_path))
        )

In [ ]:
import pandas as pd

train_labels = pd.read_excel(
    "/content/drive/MyDrive/SkincareDataset/skin_type_classification_dataset/skinalaysis_labeling_train1.xlsx"
)

print(train_labels.shape)

print(train_labels.columns.tolist())

train_labels.head()

In [ ]:
print(train_labels.columns.tolist())

In [ ]:
import torch

class_counts = [741, 874, 726, 225]

total = sum(class_counts)

weights = [
    total/(4*c)
    for c in class_counts
]

print(weights)

In [ ]:
from torchvision import datasets

train_dataset = datasets.ImageFolder(
    root="/content/drive/MyDrive/SkincareDataset/skin_type_classification_dataset/train"
)

print(train_dataset.class_to_idx)

In [ ]:
from collections import Counter

class_counts = Counter(train_dataset.targets)

print("Class Counts:")
for class_name, idx in train_dataset.class_to_idx.items():
    print(f"{class_name}: {class_counts[idx]}")

In [ ]:
import torch

num_classes = len(class_counts)
total_samples = sum(class_counts.values())

weights = []

for i in range(num_classes):
    weight = total_samples / (num_classes * class_counts[i])
    weights.append(weight)

weights = torch.tensor(weights, dtype=torch.float)

print("Class Weights:")
print(weights)

In [ ]:
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

weights = weights.to(device)

criterion = nn.CrossEntropyLoss(
    weight=weights
)

In [ ]:
print("Class Mapping:")
print(train_dataset.class_to_idx)

print("\nWeights:")
for class_name, idx in train_dataset.class_to_idx.items():
    print(f"{class_name}: {weights[idx].item():.4f}")

In [ ]:
import os

for split in ["train","valid","test"]:
    print(f"\n{split.upper()}")

    split_path = os.path.join(dataset_path, split)

    for cls in os.listdir(split_path):
        cls_path = os.path.join(split_path, cls)

        if os.path.isdir(cls_path):
            print(cls, len(os.listdir(cls_path)))

In [ ]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

In [ ]:
from PIL import Image
import os

count = 0

for root, dirs, files in os.walk(dataset_path):

    for file in files:

        if file.lower().endswith((".jpg",".jpeg",".png")):

            img = Image.open(os.path.join(root,file))

            print(img.size)

            count += 1

            if count == 10:
                break

    if count == 10:
        break

In [ ]:
import torch

print(torch.cuda.is_available())

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

In [ ]:
from torchvision import transforms

train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2
    ),
    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

valid_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

In [ ]:
from torchvision import datasets

train_dataset = datasets.ImageFolder(
    root=f"{dataset_path}/train",
    transform=train_transform
)

valid_dataset = datasets.ImageFolder(
    root=f"{dataset_path}/valid",
    transform=valid_transform
)

test_dataset = datasets.ImageFolder(
    root=f"{dataset_path}/test",
    transform=valid_transform
)

In [ ]:
print("Class Mapping:")
print(train_dataset.class_to_idx)

print("\nNumber of Classes:")
print(len(train_dataset.classes))

print("\nClass Names:")
print(train_dataset.classes)

In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2
)

In [ ]:
images, labels = next(iter(train_loader))

print("Image Batch Shape:", images.shape)
print("Label Batch Shape:", labels.shape)

In [ ]:
import matplotlib.pyplot as plt
import torchvision

images, labels = next(iter(train_loader))

grid = torchvision.utils.make_grid(images[:8], nrow=4)

plt.figure(figsize=(10,6))
plt.imshow(grid.permute(1,2,0))
plt.axis("off")
plt.show()

In [ ]:
from collections import Counter
import torch

class_counts = Counter(train_dataset.targets)

print(class_counts)

num_classes = len(class_counts)
total_samples = sum(class_counts.values())

weights = []

for i in range(num_classes):
    weight = total_samples / (num_classes * class_counts[i])
    weights.append(weight)

weights = torch.tensor(weights, dtype=torch.float)

print(weights)

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

weights = weights.to(device)

print(device)

In [ ]:
from torchvision.models import vit_b_16
from torchvision.models import ViT_B_16_Weights

weights_pretrained = ViT_B_16_Weights.DEFAULT

model = vit_b_16(
    weights=weights_pretrained
)

In [ ]:
import torch.nn as nn

model.heads.head = nn.Linear(
    model.heads.head.in_features,
    4
)

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

model = model.to(device)

print(device)
print(model)

In [ ]:
import torch.nn as nn

criterion = nn.CrossEntropyLoss(
    weight=weights
)

In [ ]:
import torch.optim as optim

optimizer = optim.AdamW(
    model.parameters(),
    lr=1e-4
)

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):

    model.train()

    running_loss = 0
    correct = 0
    total = 0

    for images, labels in loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        _, predicted = outputs.max(1)

        total += labels.size(0)

        correct += predicted.eq(labels).sum().item()

    epoch_loss = running_loss / len(loader)

    epoch_acc = 100 * correct / total

    return epoch_loss, epoch_acc

In [ ]:
def validate(model, loader, criterion, device):

    model.eval()

    running_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            running_loss += loss.item()

            _, predicted = outputs.max(1)

            total += labels.size(0)

            correct += predicted.eq(labels).sum().item()

    val_loss = running_loss / len(loader)

    val_acc = 100 * correct / total

    return val_loss, val_acc

In [ ]:
epochs = 10

for epoch in range(epochs):

    train_loss, train_acc = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        device
    )

    val_loss, val_acc = validate(
        model,
        valid_loader,
        criterion,
        device
    )

    print(
        f"Epoch {epoch+1}/{epochs}"
    )

    print(
        f"Train Loss: {train_loss:.4f}"
    )

    print(
        f"Train Acc: {train_acc:.2f}%"
    )

    print(
        f"Val Loss: {val_loss:.4f}"
    )

    print(
        f"Val Acc: {val_acc:.2f}%"
    )

    print("-"*50)

In [ ]:
from sklearn.metrics import classification_report
import torch

all_preds = []
all_labels = []

model.eval()

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)

        outputs = model(images)

        _, preds = torch.max(outputs, 1)

        all_preds.extend(preds.cpu().numpy())

        all_labels.extend(labels.numpy())

print(
    classification_report(
        all_labels,
        all_preds,
        target_names=train_dataset.classes
    )
)

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(8,6))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    xticklabels=train_dataset.classes,
    yticklabels=train_dataset.classes
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")

plt.show()

In [ ]:
print(train_dataset.class_to_idx)

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),

    transforms.ColorJitter(
        brightness=0.3,
        contrast=0.3,
        saturation=0.3
    ),

    transforms.RandomAffine(
        degrees=0,
        translate=(0.05,0.05)
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

NameError: name 'transforms' is not defined

In [ ]:
epochs = 25

for epoch in range(epochs):

    train_loss, train_acc = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        device
    )

    val_loss, val_acc = validate(
        model,
        valid_loader,
        criterion,
        device
    )

    print(
        f"Epoch {epoch+1}/{epochs}"
    )

    print(
        f"Train Loss: {train_loss:.4f}"
    )

    print(
        f"Train Acc: {train_acc:.2f}%"
    )

    print(
        f"Val Loss: {val_loss:.4f}"
    )

    print(
        f"Val Acc: {val_acc:.2f}%"
    )

    print("-"*50)

In [ ]:
import torch.optim as optim

optimizer = optim.AdamW(
    model.parameters(),
    lr=1e-4
)

In [ ]:
import os
import torch

model_path = "best_vit_skin_model.pth"

if os.path.exists(model_path):
    model.load_state_dict(
        torch.load(model_path,
        map_location=device)
    )
    model.eval()
    print("Best model loaded")
else:
    print(f"Error: Model file '{model_path}' not found.")
    print("Please ensure the model was saved during training. For example, add `torch.save(model.state_dict(), 'best_vit_skin_model.pth')` after your training loop.")

In [ ]:
test_loss, test_acc = validate(
    model,
    test_loader,
    criterion,
    device
)

print(f"Test Accuracy: {test_acc:.2f}%")

In [ ]:
from sklearn.metrics import classification_report

all_preds = []
all_labels = []

model.eval()

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)

        outputs = model(images)

        _, preds = torch.max(outputs, 1)

        all_preds.extend(preds.cpu().numpy())

        all_labels.extend(labels.numpy())

print(
    classification_report(
        all_labels,
        all_preds,
        target_names=train_dataset.classes
    )
)

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(
    all_labels,
    all_preds
)

plt.figure(figsize=(8,6))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=train_dataset.classes,
    yticklabels=train_dataset.classes
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Final Confusion Matrix")

plt.show()

In [ ]:
recommendations = {

    "dry": {

        "products": [
            "Cetaphil Moisturizing Cream",
            "Dove Gentle Cleanser",
            "Minimalist SPF 50 Sunscreen"
        ],

        "foods": [
            "Avocado",
            "Almonds",
            "Walnuts",
            "Cucumber",
            "Watermelon"
        ]
    },

    "oily": {

        "products": [
            "Sebamed Clear Face Gel",
            "Simple Face Wash",
            "Aqualogica Sunscreen"
        ],

        "foods": [
            "Orange",
            "Tomato",
            "Spinach",
            "Green Tea"
        ]
    },

    "normal": {

        "products": [
            "Cetaphil Cleanser",
            "Light Moisturizer",
            "Neutrogena Sunscreen"
        ],

        "foods": [
            "Apple",
            "Banana",
            "Carrot",
            "Yogurt"
        ]
    },

    "combination": {

        "products": [
            "CeraVe Cleanser",
            "Lightweight Moisturizer",
            "La Shield Sunscreen"
        ],

        "foods": [
            "Berries",
            "Cucumber",
            "Leafy Greens",
            "Nuts"
        ]
    }
}

In [ ]:
from PIL import Image
import torch

classes = [
    "combination",
    "dry",
    "normal",
    "oily"
]

def predict_skin_type(image_path):

    image = Image.open(image_path).convert("RGB")

    image = valid_transform(image)

    image = image.unsqueeze(0).to(device)

    model.eval()

    with torch.no_grad():

        outputs = model(image)

        _, pred = torch.max(outputs,1)

    return classes[pred.item()]

In [ ]:
def get_recommendations(skin_type):

    return recommendations[skin_type]

In [ ]:
skin_type = predict_skin_type(
    f"{dataset_path}/test/normal/normal_64c017e2b0f059a5db75_jpg.rf.3ea618ed3fd3fad5f31aed80ad19de3c.jpg"
)

print("Skin Type:", skin_type)

result = get_recommendations(
    skin_type
)

print("\nProducts:")
for item in result["products"]:
    print("-", item)

print("\nFoods:")
for item in result["foods"]:
    print("-", item)

NameError: name 'valid_transform' is not defined

In [ ]:
from PIL import Image
import torch
import torch.nn.functional as F

classes = [
    "combination",
    "dry",
    "normal",
    "oily"
]

def predict_skin_type(image_path):

    image = Image.open(image_path).convert("RGB")

    image = valid_transform(image)

    image = image.unsqueeze(0).to(device)

    model.eval()

    with torch.no_grad():

        outputs = model(image)

        probs = F.softmax(outputs, dim=1)

        confidence, pred = torch.max(probs, 1)

    return (
        classes[pred.item()],
        confidence.item() * 100
    )

In [ ]:
skin_type, confidence = predict_skin_type(
     f"{dataset_path}/test/normal/normal_64c017e2b0f059a5db75_jpg.rf.3ea618ed3fd3fad5f31aed80ad19de3c.jpg"
)

print("Predicted Skin Type:", skin_type)
print("Confidence:", round(confidence,2), "%")

NameError: name 'valid_transform' is not defined

In [ ]:
def recommend(image_path):

    skin_type, confidence = predict_skin_type(
        image_path
    )

    result = recommendations[skin_type]

    print("="*50)

    print("Skin Type :", skin_type)

    print(
        "Confidence :",
        round(confidence,2),
        "%"
    )

    print("\nRecommended Products:")

    for product in result["products"]:
        print("•", product)

    print("\nRecommended Foods:")

    for food in result["foods"]:
        print("•", food)

    print("="*50)

In [ ]:
import os

dataset_path = "/content/drive/MyDrive/SkincareDataset/skin_type_classification_dataset"

# Construct the path to the 'normal' class within the 'test' split
normal_test_dir = os.path.join(dataset_path, "test", "normal")

# Check if the directory exists
if not os.path.isdir(normal_test_dir):
    print(f"Error: Directory not found: {normal_test_dir}. Please verify your dataset structure.")
else:
    # List all files in the directory
    files_in_dir = os.listdir(normal_test_dir)

    # Filter for image files (assuming .jpg, .jpeg, .png)
    image_files = [f for f in files_in_dir if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

    if image_files:
        # Use the first image file found for demonstration
        first_image_path = os.path.join(normal_test_dir, image_files[0])
        print(f"Using image for recommendation: {first_image_path}")
        recommend(first_image_path)
    else:
        print(f"No image files found in {normal_test_dir}. Please ensure there are images to process.")

Using image for recommendation: /content/drive/MyDrive/SkincareDataset/skin_type_classification_dataset/test/normal/normal_059cf2165bd4da06cc0f_jpg.rf.3a94f4a0db8f9dc36629a1cadc707d80.jpg


NameError: name 'valid_transform' is not defined

In [ ]:
import torch
import torch.nn as nn
from torchvision.models import vit_b_16, ViT_B_16_Weights
import os

# Define device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Re-initialize the model (as it might not be in scope)
weights_pretrained = ViT_B_16_Weights.DEFAULT
model = vit_b_16(
    weights=weights_pretrained
)

# Re-apply the head modification
model.heads.head = nn.Linear(
    model.heads.head.in_features,
    4
)

# Move model to device
model = model.to(device)

# Define the model save path
model_save_path = "/content/drive/MyDrive/SkincareDataset/final_vit_skin_model.pth"

# Ensure the parent directory exists
os.makedirs(os.path.dirname(model_save_path), exist_ok=True)

torch.save(
    model.state_dict(),
    model_save_path
)

print("Model Saved Successfully")

Model Saved Successfully


In [ ]:
import pandas as pd

history = pd.DataFrame(
    columns=[
        "Image",
        "Skin_Type",
        "Confidence"
    ]
)

history.to_csv(
    "prediction_history.csv",
    index=False
)

In [ ]:
!pip install -q gradio

In [ ]:
custom_css = """
.gradio-container{
    max-width:1400px !important;
    margin:auto;
    font-family:'Segoe UI',sans-serif;
}

.main-title{
    text-align:center;
    font-size:42px;
    font-weight:700;
    color:#4F46E5;
    margin-bottom:5px;
}

.sub-title{
    text-align:center;
    font-size:18px;
    color:#64748B;
    margin-bottom:25px;
}

.card{
    border-radius:16px;
    padding:12px;
    box-shadow:0px 2px 10px rgba(0,0,0,0.08);
}

.footer{
    text-align:center;
    color:gray;
    font-size:14px;
}
"""

In [ ]:
import torch
import torch.nn.functional as F
from PIL import Image

classes = [
    "combination",
    "dry",
    "normal",
    "oily"
]

def predict_and_recommend(image):

    image = image.convert("RGB")

    image = valid_transform(image)

    image = image.unsqueeze(0).to(device)

    model.eval()

    with torch.no_grad():

        outputs = model(image)

        probs = F.softmax(outputs, dim=1)

        confidence, pred = torch.max(probs, 1)

    skin_type = classes[pred.item()]

    confidence_score = round(
        confidence.item()*100,
        2
    )

    products = "\n".join(
    recommendations[skin_type]["products"]
    )

    foods = "\n".join(
    recommendations[skin_type]["foods"]
    )

    # Using generate_insight from cell Cv0P34rg5W7y
    advice = generate_insight(skin_type)

    return (
        skin_type,
        confidence_score,
        f"{confidence_score:.2f}%",
        products,
        foods,
        advice
    )

In [ ]:
def generate_insight(skin_type):

    insights = {

        "dry":
        """
        🔎 AI Skin Assessment

        • Low moisture retention detected
        • Skin barrier may be weakened
        • Higher chance of flakiness

        Recommendations:
        • Use cream-based moisturizers
        • Avoid harsh cleansers
        • Increase water intake

        Expected Outcome:
        • Improved hydration
        • Better skin texture
        """,

        "oily":
        """
        🔎 AI Skin Assessment

        • Excess sebum production detected
        • Moderate acne susceptibility

        Recommendations:
        • Oil-free cleanser
        • Lightweight moisturizer
        • SPF protection

        Expected Outcome:
        • Reduced oiliness
        • Clearer skin appearance
        """,

        "normal":
        """
        🔎 AI Skin Assessment

        • Balanced hydration levels
        • Healthy skin barrier

        Recommendations:
        • Maintain current routine
        • Daily sunscreen
        • Balanced nutrition

        Expected Outcome:
        • Consistent skin health
        """,

        "combination":
        """
        🔎 AI Skin Assessment

        • Oily T-zone detected
        • Dryness in other regions

        Recommendations:
        • Gentle cleanser
        • Gel moisturizer
        • Spot-based skincare

        Expected Outcome:
        • Better oil balance
        """
    }

    return insights.get(skin_type.lower(),"No insight available.")

In [ ]:
!pip uninstall -y gradio
!pip install -q gradio
import gradio as gr

custom_css = """
body{
    background:#F5F7FA;
}

.gradio-container{
    max-width:1400px !important;
    margin:auto;
    font-family:'Segoe UI',sans-serif;
}

.main-title{
    text-align:center;
    font-size:42px;
    font-weight:700;
    color:#0F766E;
    margin-top:10px;
}

.sub-title{
    text-align:center;
    font-size:18px;
    color:#64748B;
    margin-bottom:25px;
}

.result-card{
    background:white;
    border-radius:16px;
    padding:15px;
    border:1px solid #E5E7EB;
}

.footer{
    text-align:center;
    color:#6B7280;
    font-size:14px;
    padding:20px;
}
"""

with gr.Blocks(
    css=custom_css,
    theme=gr.themes.Soft(
        primary_hue="emerald",
        secondary_hue="cyan"
    ),
    title="Derma-Nutri"
) as demo:

    gr.HTML("""
    <div class='main-title'>
        🌿 Derma-Nutri
    </div>

    <div class='sub-title'>
        AI Powered Skin Analysis, Skincare &
        Nutrition Recommendation System
    </div>
    """)

    with gr.Row():

        # LEFT PANEL

        with gr.Column(scale=1):

            image_input = gr.Image(
                type="pil",
                label="Upload Facial Image",
                height=550
            )

            analyze_btn = gr.Button(
                "Analyze Skin",
                variant="primary",
                size="lg"
            )

        # RIGHT PANEL

        with gr.Column(scale=1):

            skin_output = gr.Textbox(
                label="Predicted Skin Type"
            )

            confidence_bar = gr.Slider(
                minimum=0,
                maximum=100,
                value=0,
                interactive=False,
                label="AI Confidence"
            )

            confidence_output = gr.Textbox(
                label="Confidence Score (%)"
            )

            product_output = gr.Textbox(
                label="Recommended Products",
                lines=5
            )

            food_output = gr.Textbox(
                label="Recommended Foods",
                lines=5
            )

            insight_output = gr.Textbox(
                label="AI Dermatology Insight",
                lines=8
            )

    analyze_btn.click(
        fn=predict_and_recommend,
        inputs=image_input,
        outputs=[
            skin_output,
            confidence_bar,
            confidence_output,
            product_output,
            food_output,
            insight_output
        ]
    )

    gr.Markdown("---")

    gr.HTML("""
    <div style="
background:#1F2937;
border:1px solid #374151;
border-radius:12px;
padding:20px;
margin-top:15px;
color:white;
">

<h2 style="
color:#10B981;
margin-bottom:15px;
">
📊 Model Information
</h2>

<table style="
width:100%;
font-size:15px;
border-collapse:collapse;
color:white;
">

<tr>
<td style="padding:10px;"><b>Model</b></td>
<td style="padding:10px;">Vision Transformer (ViT)</td>
</tr>

<tr>
<td style="padding:10px;"><b>Classes</b></td>
<td style="padding:10px;">Dry, Oily, Normal, Combination</td>
</tr>

<tr>
<td style="padding:10px;"><b>Test Accuracy</b></td>
<td style="padding:10px;">81.38%</td>
</tr>

<tr>
<td style="padding:10px;"><b>Optimizer</b></td>
<td style="padding:10px;">AdamW</td>
</tr>

<tr>
<td style="padding:10px;"><b>Dataset</b></td>
<td style="padding:10px;">Facial Skin Analysis & Type Classification</td>
</tr>

<tr>
<td style="padding:10px;"><b>Images Used</b></td>
<td style="padding:10px;">2800+</td>
</tr>

</table>

</div>
""")

gr.HTML("""
<div style="
text-align:center;
margin-top:15px;
padding:10px;
color:#9CA3AF;
font-size:13px;
">

<b>Derma-Nutri</b><br>

AI Powered Personalized Skincare &
Nutrition Recommendation System

</div>
""")

demo.launch(share=True)

Found existing installation: gradio 5.50.0
Uninstalling gradio-5.50.0:
  Successfully uninstalled gradio-5.50.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 70.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.6/73.6 kB 6.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires starlette<1.0.0,>=0.49.1, but you have starlette 1.3.1 which is incompatible.


/tmp/ipykernel_5924/995398462.py:46: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_5924/995398462.py:46: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ce53a6da5233081130.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


AttributeError: module 'gradio' has no attribute 'blocks'

In [ ]:
import gradio as gr

custom_css = """
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&display=swap');

body{
font-family:'Inter',sans-serif;
}

.gradio-container{
max-width:1400px !important;
margin:auto !important;
background:#0F172A;
}

h1,h2,h3{
font-family:'Inter',sans-serif;
}

.output-box textarea{
font-size:15px !important;
}

footer{
display:none !important;
}
"""

with gr.Blocks(
theme=gr.themes.Soft(
primary_hue="slate",
secondary_hue="gray"
),
css=custom_css,
title="Derma-Nutri"
) as demo:

    gr.HTML("""

<div style="
text-align:center;
padding:15px;
margin-bottom:10px;
">

<h1 style="
color:#14B8A6;
font-size:48px;
font-weight:700;
margin-bottom:8px;
">
Derma-Nutri
</h1>

<p style="
color:#94A3B8;
font-size:18px;
">
AI Powered Skin Analysis, Skincare & Nutrition Recommendation System
</p>

</div>

""")

    with gr.Row(equal_height=True):

        # LEFT PANEL

        with gr.Column(scale=5):

            gr.Markdown("### Upload Facial Image")

            image_input = gr.Image(
                type="pil",
                height=520,
                show_download_button=False,
                show_share_button=False
            )

            analyze_btn = gr.Button(
                "Analyze Skin",
                variant="primary",
                size="lg"
            )

        # RIGHT PANEL

        with gr.Column(scale=5):

            gr.Markdown("### Prediction Results")

            skin_type = gr.Textbox(
                label="Predicted Skin Type",
                lines=1,
                interactive=False
            )

            confidence_meter = gr.Slider(
                minimum=0,
                maximum=100,
                value=0,
                interactive=False,
                label="AI Confidence Meter"
            )

            confidence_score = gr.Textbox(
                label="Confidence Score (%)",
                lines=1,
                interactive=False
            )

            products = gr.Textbox(
                label="Recommended Products",
                lines=4,
                interactive=False
            )

            foods = gr.Textbox(
                label="Recommended Foods",
                lines=4,
                interactive=False
            )

            insight = gr.Textbox(
                label="AI Dermatology Insight",
                lines=8,
                interactive=False
            )

    analyze_btn.click(
        fn=predict_and_recommend,
        inputs=image_input,
        outputs=[
            skin_type,
            confidence_meter,
            confidence_score,
            products,
            foods,
            insight
        ]
    )

demo.launch(
debug=True,
share=True
)

In [ ]:
import gradio as gr

with gr.Blocks(
    title="Derma-Nutri",
    css=custom_css,
    theme=gr.themes.Base()
) as demo:

    gr.HTML("""
    <div style='text-align:center;padding:10px;'>

    <h1 style='
    color:#F8FAFC;
    font-size:52px;
    font-weight:700;
    margin-bottom:5px;
    '>
    Derma-Nutri
    </h1>

    <p style='
    color:#94A3B8;
    font-size:18px;
    margin-top:0px;
    '>
    AI Powered Skin Analysis, Skincare & Nutrition Recommendation System
    </p>

    </div>
    """)

    with gr.Row(equal_height=True):

        # LEFT PANEL

        with gr.Column(scale=5):

            gr.Markdown("### Upload Facial Image")

            image_input = gr.Image(
                type="pil",
                height=500,
                show_download_button=False,
                show_share_button=False
            )

            analyze_btn = gr.Button(
                "Analyze Skin",
                variant="primary",
                size="lg"
            )

        # RIGHT PANEL

        with gr.Column(scale=5):

            gr.Markdown("### Prediction Results")

            skin_type = gr.Textbox(
                label="Predicted Skin Type",
                lines=1,
                interactive=False
            )

            confidence_meter = gr.Slider(
                minimum=0,
                maximum=100,
                value=0,
                interactive=False,
                label="AI Confidence Meter"
            )

            confidence_score = gr.Textbox(
                label="Confidence Score (%)",
                lines=1,
                interactive=False
            )

            products = gr.Textbox(
                label="Recommended Products",
                lines=4,
                interactive=False
            )

            foods = gr.Textbox(
                label="Recommended Foods",
                lines=4,
                interactive=False
            )

            insight = gr.Textbox(
                label="AI Dermatology Insight",
                lines=8,
                interactive=False
            )

    analyze_btn.click(
        fn=predict_and_recommend,
        inputs=image_input,
        outputs=[
            skin_type,
            confidence_meter,
            confidence_score,
            products,
            foods,
            insight
        ]
    )

demo.launch(
    debug=True,
    share=True
)

In [ ]:
import gradio as gr

with gr.Blocks(
    css=custom_css,
    theme=gr.themes.Soft()
) as demo:

    gr.HTML("""
    <div style='text-align:center;padding:20px;'>

    <h1>Derma-Nutri</h1>

    <p style='color:#94a3b8;font-size:18px'>
    AI Powered Skin Analysis, Skincare & Nutrition Recommendation System
    </p>

    </div>
    """)

    with gr.Row():

        # LEFT SIDE
        with gr.Column(scale=5):

            gr.Markdown("## Upload Facial Image")

            image_input = gr.Image(
                type="pil",
                height=600
            )

        # RIGHT SIDE
        with gr.Column(scale=5):

            gr.Markdown("## Prediction Results")

            skin_type = gr.Textbox(
                label="Predicted Skin Type"
            )

            confidence_slider = gr.Slider(
                minimum=0,
                maximum=100,
                label="AI Confidence Meter",
                interactive=False
            )

            confidence_text = gr.Textbox(
                label="Confidence Score (%)"
            )

            products = gr.Textbox(
                label="Recommended Products",
                lines=4
            )

            foods = gr.Textbox(
                label="Recommended Foods",
                lines=4
            )

            insight = gr.Textbox(
                label="AI Dermatology Insight",
                lines=8
            )

    analyze_btn = gr.Button(
        "Analyze Skin",
        variant="primary",
        size="lg"
    )

    analyze_btn.click(
        fn=predict_and_recommend,
        inputs=image_input,
        outputs=[
            skin_type,
            confidence_slider,
            confidence_text,
            products,
            foods,
            insight
        ]
    )

    gr.HTML("""
    <hr style='margin-top:30px'>

    <div style='text-align:center;color:#94a3b8'>

    <h3>Model Information</h3>

    <p>Vision Transformer (ViT)</p>

    <p>Classes: Dry, Oily, Normal, Combination</p>

    <p>Test Accuracy: 81.38%</p>

    <p>Optimizer: AdamW</p>

    <br>

    © 2026 Derma-Nutri

    </div>
    """)

demo.launch(share=True)

In [ ]:
import gradio as gr

with gr.Blocks(
    title="Derma-Nutri",
    theme=gr.themes.Soft()
) as demo:

    gr.HTML("""
    <div style="text-align:center;margin-bottom:20px;">
        <h1 style="
            font-size:52px;
            font-weight:700;
            color:#14b8a6;
            margin-bottom:5px;">
            🌿 Derma-Nutri
        </h1>

        <p style="
            font-size:20px;
            color:#94a3b8;">
            AI Powered Skin Analysis, Skincare & Nutrition Recommendation System
        </p>
    </div>
    """)

    with gr.Row(equal_height=True):

        # LEFT SIDE
        with gr.Column(scale=5):

            image_input = gr.Image(
                type="pil",
                label="Upload Facial Image",
                height=520
            )

            analyze_btn = gr.Button(
                "🔍 Analyze Skin",
                variant="primary",
                size="lg"
            )

        # RIGHT SIDE
        with gr.Column(scale=5):

            skin_type = gr.Textbox(
                label="Predicted Skin Type"
            )

            confidence_meter = gr.Slider(
                minimum=0,
                maximum=100,
                interactive=False,
                label="AI Confidence Meter"
            )

            confidence_score = gr.Textbox(
                label="Confidence Score (%)"
            )

            products = gr.Textbox(
                label="Recommended Products",
                lines=4
            )

            foods = gr.Textbox(
                label="Recommended Foods",
                lines=4
            )

            insights = gr.Textbox(
                label="AI Dermatology Insight",
                lines=10
            )

    analyze_btn.click(
        fn=predict_and_recommend,
        inputs=image_input,
        outputs=[
            skin_type,
            confidence_meter,
            confidence_score,
            products,
            foods,
            insights
        ]
    )

demo.launch(share=True)

/tmp/ipykernel_5924/1243690529.py:3: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8cd67bd2890aadee26.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


AttributeError: module 'gradio' has no attribute 'blocks'